In [2]:
#!pip install yfinance

In [3]:
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from typing import Callable, Dict, List, Tuple
import yfinance as yf

In [7]:
def load_price_data(path: str) -> pd.DataFrame:
    """
    Load price data from CSV with at least:
        - Date
        - Close price
    """

    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').set_index('Date')
    df = df[['Close']].astype(float)
    return df

In [ ]:
def strategy_buy_and_hold(df: pd.DataFrame) -> pd.Series:
    """
    Always long (after first day)
    """

    signal = pd.Series(1, index=df.index)
    signal.iloc[0] = 0 # start flat on the first bar
    return signal

In [ ]:
def strategy_sma_long_only(df: pd.DataFrame,
                           slow_window: int = 20,
                           long_window: int = 50) -> pd.Series:
    """
    Long-only SMA crossover:
        - Long when fast SMA > slow SMA
        - Flat otherwise
    """

    tmp = df.copy()
    tmp['SMA_fast'] = tmp['Close'].rolling(window=long_window).mean()
    tmp['SMA_slow'] = tmp['Close'].rolling(window=slow_window).mean()

    signal = (tmp['SMA_fast'] > tmp['SMA_slow']).astype(int)

    # Shift by 1 bar to avoid lookahead bias
    signal = signal.shift(1).fillna(0)
    return signal

In [ ]:
def strategy_sma_long_short(df: pd.DataFrame,
                            fast_window: int = 20,
                            slow_window: int = 50) -> pd.Series:
    """
    Long/short SMA crossover:
        - Long (1) when fast SMA > slow SMA
        - Short (-1) when fast SMA < slow SMA
        - Flat (0) when equal or not enough data
    """

    tmp = df.copy()
    tmp['SMA_fast'] = tmp['Close'].rolling(window=fast_window).mean()
    tmp['SMA_slow'] = tmp['Close'].rolling(window=slow_window).mean()

    signal = pd.Series(0, index=df.index)
    signal[(tmp['SMA_fast'] > tmp['SMA_slow'])] = 1
    signal[(tmp['SMA_fast'] < tmp['SMA_slow'])] = -1

    # Shift by 1 bar to avoid lookahead bias
    signal = signal.shift(1).fillna(0)
    return signal


In [ ]:
def strategy_mean_reversion_long_short(df: pd.DataFrame,
                                       lookback: int = 5,
                                       z_entry: float = 1.0) -> pd.Series:
    """
    Simple mean-reversion on daily returns with long/short:
        - If today's return is more than z_entry std BELOW rolling mean -> long next bar
        - If today's return is more than z_entry std ABOVE rolling mean -> short next bar
        - Else flat
    """

    tmp = df.copy()
    tmp['ret'] = tmp['Close'].pct_change()

    mu = tmp['ret'].rolling(lookback).mean()
    sigma = tmp['ret'].rolling(lookback).std()
    z = (tmp['ret'] - mu) / sigma

    signal = pd.Series(0, index=tmp.index)
    signal[(z < -z_entry)] = 1  # big down move -> long
    signal[(z > z_entry)] = -1  # big up move -> short

    signal = signal.shift(1).fillna(0)
    return signal

In [11]:
def backtest(price_df: pd.DataFrame,
             initial_capital: float = 10_0000.0,
             trading_fee_bps: float = 10.0) -> pd.DataFrame:
    """
    Generic backtest for a given signal.
    signal: Series aligned to price_df.index, values in {-1, 0, 1}
        1 = long, -1 = short, 0 = no position

    Assumes we put 100% of equity into the position (no leverage)

    Args:
        df (pd.DataFrame): DataFrame containing 'Close' prices and 'signal'.
        initial_capital (float): Starting cash for the backtest.
        trading_fee_bps (float): Transaction cost per trade as a percentage
          (e.g., 0.001 for 0.1%).

    Returns:
        pd.DataFrame: DataFrame with daily returns, strategy returns,
        and cumulative returns.
    """

    df = price_df.copy()

    # Align signal and ensure float
    df['signal'] = signal.reindex(df.index).fillna(0).astype(float)


    # Calculate daily percentage change in Close price
    df['asset_ret'] = df['Close'].pct_change().fillna(0.0)

    # Postition is the signal (0 or 1)
    df['position'] = df['signal']

    # When position changes, we "trade"
    df['position_change'] = df['position'].diff().fillna(df['position'])

    fee_rate = trading_fee_bps / 10_0000.0  # bps -> decimal

    equity = initial_capital
    equity_series = []

    for i, row in df.iterrows():
        pos = row['position']
        asset_ret = row['asset_ret']

        # PnL: long/short exposure * return
        # long (1): equity *= (1 + r)
        # short (-1): equity *= (1 - r)

        # Trading cost when position changes (including flip long <-> short)
        if row['position_change'] != 0:
            # abs(postition_change) = 1 for open/close, 2 for flip long <-> short
            traded_notional = abs(row['position_change']) * equity
            cost = traded_notional * fee_rate
            equity -= cost

        equity_series.append(equity)

    df['equity'] = equity_series
    df['strategy_ret'] = df['equity'].pct_change().fillna(0.0)
    df['cum_ret'] = (1 + df['strategy_ret']).cumprod()

    return df

In [10]:
def add_signals(df: pd.DataFrame,
                fast_window: int = 20,
                slow_window: int = 50) -> pd.DataFrame:
    """
    Add moving average signals.
    Signal is:
        1 -> long
        0 -> flat
        -1 -> short
    """

    df = df.copy()
    df['fast_ma'] = df['Close'].rolling(fast_window).mean()
    df['slow_ma'] = df['Close'].rolling(slow_window).mean()

    # Basic rule: long when fast > slow, else flat
    df['signal'] = 0.0
    df.loc[df['fast_ma'] > df['slow_ma'], 'signal'] = 1.0
    df.loc[df['fast_ma'] < df['slow_ma'], 'signal'] = -1.0

    # Shift signal by 1 bar to avoid lookahead bias
    df['signal'] = df['signal'].shift(1).fillna(0)
    return df

In [ ]:
def performance_summary(df: pd.DataFrame,
                        risk_free_rate: float = 0.0) -> Dict[str, float]
    """
    Basic performance stats from an equity curve.
    """

    trading_days = 252
    rets = df["strategty_ret"]

    cum_return = (1 + rets).prod() - 1

    if len(rets) > 0:
        ann_return = (1 + rets.cum_return) ** (trading_days / len(rets)) - 1
    else:
        ann_return = np.nan

    ann_vol = rets.std() * np.sqrt(trading_days)

    if ann_vol > 0:
        sharpe = (ann_return - risk_free_rate) / ann_vol
    else:
        sharpe = np.nan

    equity = df['equity']
    roll_max = equity.cummax()
    drawdown = equity / roll_max - 1
    max_dd = drawdown.min()

    return {
        "Cumulative Return": cum_return,
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd,
        "Final Equity": float(equity.iloc[-1])  # Convert to float
    }

In [5]:
# Define the ticker symbol (e.g., SPY)
ticker_symbol = 'SPY'

# Define the period for historical data (e.g., '25y' for 25 years)
period = '25y'

# Fetch historical data
stock_data = yf.download(ticker_symbol, period=period)

# Display the first few rows of the data
print(f"Historical data for {ticker_symbol} for the last {period}:")
print(stock_data.sample(5))

/tmp/ipython-input-1626056231.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker_symbol, period=period)
[*********************100%***********************]  1 of 1 completed

Historical data for SPY for the last 25y:
Price            Close        High         Low        Open     Volume
Ticker             SPY         SPY         SPY         SPY        SPY
Date                                                                 
2013-01-16  117.746620  117.930784  117.394299  117.522418  104849500
2015-07-29  177.182404  177.409368  175.955059  176.097966  105791300
2004-01-16   76.257011   76.310413   75.856463   76.130171   31922700
2015-11-03  178.297867  178.855579  177.199347  177.427504   95246100
2013-12-12  144.727646  145.320755  144.427019  145.142008  115565000
